# LC 84 — Largest Rectangle in Histogram
**Difficulty:** Hard &nbsp;|&nbsp; **Category:** Stack
**Pattern:** Monotonic Increasing Stack with Start Index

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> A bar's rectangle can
extend left as far as the nearest shorter bar to
its left, and right until a shorter bar blocks it.
The monotonic stack tracks exactly those boundaries
and resolves each bar the moment it is blocked.
</div>

## Official Problem Statement

Given an array of integers `heights` representing
the histogram's bar height where the width of each
bar is `1`, return the area of the largest rectangle
in the histogram.

**Example 1:**
```
Input:  heights = [2,1,5,6,2,3]
Output: 10
Explanation: Rectangle from bars 2,3 (heights 5,6)
             has area 5*2 = 10.
```
**Example 2:**
```
Input:  heights = [2,4]
Output: 4
```

**Constraints:**
- `1 <= heights.length <= 10^5`
- `0 <= heights[i] <= 10^4`

## What This Is Actually Asking

You have a skyline of bars, each one unit wide.
Find the largest solid rectangle you can draw
that fits entirely within that skyline.
The rectangle can span multiple bars but its height
is capped by the shortest bar it covers.
Return the area of the biggest such rectangle.

## Walk Through an Example by Hand

```
heights = [2, 1, 5, 6, 2, 3]
idx:        0  1  2  3  4  5

Stack stores (start_index, height) pairs.
Pop when current height < stack top height.
The popped bar's rectangle extends from its
start_index all the way to current index.

i=0 h=2  stack empty  push (0,2)   stack=[(0,2)]
i=1 h=1  1<2 -> pop (0,2)  area=2*(1-0)=2  max=2
         start=0 (can extend left to where 2 started)
         1<=1 -> push (0,1)          stack=[(0,1)]
i=2 h=5  5>1 -> push (2,5)          stack=[(0,1),(2,5)]
i=3 h=6  6>5 -> push (3,6)          stack=[(0,1),(2,5),(3,6)]
i=4 h=2  2<6 -> pop (3,6)  area=6*(4-3)=6  max=6
         2<5 -> pop (2,5)  area=5*(4-2)=10 max=10
         start=2 (that's where 5 started)
         2>=1 -> push (2,2)          stack=[(0,1),(2,2)]
i=5 h=3  3>2 -> push (5,3)          stack=[(0,1),(2,2),(5,3)]

End of array — pop remaining:
  pop (5,3)  area=3*(6-5)=3   max=10
  pop (2,2)  area=2*(6-2)=8   max=10
  pop (0,1)  area=1*(6-0)=6   max=10

Answer: 10
```

## The Picture

```
heights = [2, 1, 5, 6, 2, 3]

  6     |  |
  5     |  |  |
  4     |  |  |
  3     |  |  |  |
  2  |  |  |  |  |
  1  |  |  |  |  |  |
     0  1  2  3  4  5

Largest rectangle: bars 2 and 3 (height 5, width 2)
  ████
  ████   <- 5 × 2 = 10
  ████
  ████
  ████
     2  3

Stack insight:
  When bar i is SHORTER than stack top:
    top bar is blocked on the right at i
    width = i - start_index (how far left it can go)
    area  = height × width

  Track start_index: when popping, the new bar
  can start from where the popped bar started
  (it was already as tall as it needs to be there).
```

## When To Use This Pattern

- When you see **largest area in a histogram or
  grid**, think **monotonic increasing stack with
  (start_index, height) pairs**
- When a shorter bar blocks a taller one on the
  right, think **pop and compute area now**
- When the popped bar could have extended left
  into taller bars before it, think **inherit
  start_index from the popped bar**
- When bars remain after the loop, think **they
  extend all the way to len(heights)**

## The Approach

Walk through the histogram with a stack storing
(start_index, height) pairs in increasing height
order.
When a bar shorter than the stack top arrives,
pop and calculate the area using the popped height
and the width from the popped start_index to the
current index; the new bar inherits the start_index
of the last popped bar.
After the full array, pop all remaining bars using
the array length as the right boundary.

In [20]:
from typing import List  # type hints for the solution

In [21]:
def test_harness(func):
    tests = [
        # (heights, expected)
        ([2,1,5,6,2,3],  10),
        ([2,4],           4),
        ([1],             1),   # single bar
        ([0],             0),   # zero height
        ([6,2,5,4,5,1,6], 12), # classic
        ([2,2,2,2],       8),   # all equal
        ([1,2,3,4,5],    9),   # ascending
        ([5,4,3,2,1],    9),   # descending
        ([4,2,0,3,2,5],  6),
        ([3,6,5,7,4,8,1,0], 20),
    ]

    passed = 0
    for i, (heights, expected) in enumerate(tests):
        result = func(heights[:])
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"Test {i+1}: {status} | "
            f"heights={heights} | "
            f"expected={expected} | got={result}"
        )

    print(f"\n{passed}/{len(tests)} tests passed")

In [41]:
def largestRectangleArea(heights: List[int]) -> int:
    """
    Return largest rectangle area in the histogram.

    Monotonic increasing stack of (start, height).
    When heights[i] < stack top: pop, area = height *
    (i - start). New bar inherits popped start_index.
    After loop, pop remaining bars using len(heights)
    as the right boundary.

    Time:  O(n) — each bar pushed and popped once
    Space: O(n) — stack holds at most n entries
    """
    # We should traverse from left to right.
    # at each stage... if our bar is longer than the stack current top store the index, height 
    # of the current back on top of stack (append call)
    # if other wise. loop popping items from stack calculating rectangle are as height (curindex, popped index)
    # keep caching the last index popped
    # when all longer than current popped .. store current as cacshed index and height.
    # Once the traverse finished.. Clear the stack calculating rectangle area as height * len(heights) - cashed index
    # return max area found
     
    stack, cached_index, res = [] , 0 , 0
    for r, h in enumerate(heights):
        cached_index = r
        while stack and h < stack[-1][1]:
            popped_index, popped_height = stack.pop()
            cached_index =  popped_index
            res = max(res, (r - popped_index) * popped_height) 
        stack.append([cached_index, h])
        
    while(stack):
        pair = stack.pop()
        res = max(res, pair[1] * (len(heights)-pair[0]) )
    return(res)
print(largestRectangleArea([2,1,5,6,2,3]))   # 10
print(largestRectangleArea([2,4]))           # 4
print(largestRectangleArea([1]))             # 1
print(largestRectangleArea([5,4,3,2,1]))     # 9
test_harness(largestRectangleArea)
    
    

10
4
1
9
Test 1: PASSED | heights=[2, 1, 5, 6, 2, 3] | expected=10 | got=10
Test 2: PASSED | heights=[2, 4] | expected=4 | got=4
Test 3: PASSED | heights=[1] | expected=1 | got=1
Test 4: PASSED | heights=[0] | expected=0 | got=0
Test 5: PASSED | heights=[6, 2, 5, 4, 5, 1, 6] | expected=12 | got=12
Test 6: PASSED | heights=[2, 2, 2, 2] | expected=8 | got=8
Test 7: PASSED | heights=[1, 2, 3, 4, 5] | expected=9 | got=9
Test 8: PASSED | heights=[5, 4, 3, 2, 1] | expected=9 | got=9
Test 9: PASSED | heights=[4, 2, 0, 3, 2, 5] | expected=6 | got=6
Test 10: PASSED | heights=[3, 6, 5, 7, 4, 8, 1, 0] | expected=20 | got=20

10/10 tests passed


In [38]:
def largestRectangleArea(heights: List[int]) -> int:
    """
    Return largest rectangle area in the histogram.

    Monotonic increasing stack of (start, height).
    When heights[i] < stack top: pop, area = height *
    (i - start). New bar inherits popped start_index.
    After loop, pop remaining bars using len(heights)
    as the right boundary.

    Time:  O(n) — each bar pushed and popped once
    Space: O(n) — stack holds at most n entries
    """
    res = 0
    stack = [] # monotonic increasing stack of index, height
               # pairs the revoke others will inherit their index
               # If upcoming heightis less than top of stack .. evict and calculate areas

    # the way in
    for r, h in enumerate(heights):
        cached_index = r
        while stack and h < stack[-1][1]:
            popped_index, popped_height = stack.pop()
            cached_index = popped_index
            res = max(res, popped_height * ( r- popped_index ) )
        stack.append ([cached_index, h])
    # clearing stack of unprocessed areas
    while stack:
        idx, h = stack.pop()
        res = max(res, h * ( len(heights) - idx) )
     
    return (res)    



# Quick debug — run this cell while building
print(largestRectangleArea([2,1,5,6,2,3]))  # 10
print(largestRectangleArea([2,4]))           # 4
print(largestRectangleArea([1]))             # 1
print(largestRectangleArea([5,4,3,2,1]))    # 9
test_harness(largestRectangleArea)

10
4
1
9
Test 1: PASSED | heights=[2, 1, 5, 6, 2, 3] | expected=10 | got=10
Test 2: PASSED | heights=[2, 4] | expected=4 | got=4
Test 3: PASSED | heights=[1] | expected=1 | got=1
Test 4: PASSED | heights=[0] | expected=0 | got=0
Test 5: PASSED | heights=[6, 2, 5, 4, 5, 1, 6] | expected=12 | got=12
Test 6: PASSED | heights=[2, 2, 2, 2] | expected=8 | got=8
Test 7: PASSED | heights=[1, 2, 3, 4, 5] | expected=9 | got=9
Test 8: PASSED | heights=[5, 4, 3, 2, 1] | expected=9 | got=9
Test 9: PASSED | heights=[4, 2, 0, 3, 2, 5] | expected=6 | got=6
Test 10: PASSED | heights=[3, 6, 5, 7, 4, 8, 1, 0] | expected=20 | got=20

10/10 tests passed


In [ ]:
# Uncomment and run when solution is ready
# test_harness(largestRectangleArea)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force — try all pairs | O(n²) | O(1) |
| Divide and conquer | O(n log n) | O(n) |
| Monotonic stack | O(n) | O(n) |

The stack is optimal — each bar is processed
exactly once (one push, one pop) regardless of
how the inner while loop distributes the work.

## Real World Connection

At Citi, capacity planning asks: what is the
largest continuous time block across a cluster
where every server has at least H% headroom?
The histogram bars are the per-server headroom
values across time slots; the largest rectangle
is the widest scheduling window guaranteed for
all servers simultaneously.
The monotonic stack finds this in a single O(n)
pass over the telemetry summary table instead
of the O(n²) pairwise scan the operations team
was previously running in Athena.
On AWS, the same approach sizes the optimal
Lambda concurrency reservation window across
a set of downstream consumers.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra